In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart


# global set client

In [2]:
# Path to your credentials JSON file
credentials_path = "read_BQ.json"

# Load credentials from the JSON file
credentials = service_account.Credentials.from_service_account_file(credentials_path)

# Initialize BigQuery client
client = bigquery.Client(credentials=credentials, project="iucc-f4d")

# Initialize an empty list to store data for the DataFrame
data = []

dataset_costs = {}  # Dictionary to store costs per dataset
combined_cost = 0   # Variable to track the combined cost

In [3]:
def Last_Hours_Experiment_Data(client, dataset_id, table_id, print_cost=False):
    query = f"""
    SELECT
        ExperimentData_Exp_name,
        ARRAY_AGG(DISTINCT SensorData_Name) AS sensor_names,
        MAX(TimeStamp) AS last_timestamp_BQ,
        CURRENT_DATETIME("Asia/Jerusalem") AS now_israel,
        TIMESTAMP_DIFF(
            TIMESTAMP(CURRENT_DATETIME("Asia/Jerusalem")),
            TIMESTAMP(MAX(TimeStamp)), -- Keep this as UTC for the diff calculation
            HOUR
        ) AS hours_since_last
    FROM
        `iucc-f4d.{dataset_id}.{table_id}` AS Table_DATA
    WHERE
        TIMESTAMP(DATETIME(TimeStamp, "Asia/Jerusalem")) >= TIMESTAMP_SUB(TIMESTAMP(CURRENT_DATETIME("Asia/Jerusalem")), INTERVAL 40 HOUR)
    GROUP BY
        ExperimentData_Exp_name
    ORDER BY
        last_timestamp_BQ DESC;
    """
    
    query_job = client.query(query)
    results = query_job.result()
    if print_cost:
        print(f"Query cost: {query_job.total_bytes_processed / 1e6} MB")
    # Convert query results to a DataFrame
    data = []
    for row in results:
        data.append({
            "dataset_id": dataset_id,
            "table_id": table_id,
            "ExperimentData_Exp_name": row.ExperimentData_Exp_name,
            "Distinct_SensorName_Last_40h": row.sensor_names,
            "Sensor_In_Last_40h": len(row.sensor_names) if row.sensor_names else 0,
            "last_timestamp": row.last_timestamp_BQ,
            "hours_passed_since_last_upload": row.hours_since_last,
            
        })
    return pd.DataFrame(data)

In [4]:
def query_unique_sensors_name(client,dataset_id, table_id,experiment_name):
    """
    Query to get unique sensor names for a specific experiment.

    Parameters:
        client (bigquery.Client): Initialized BigQuery client.
        dataset_id (str): The ID of the dataset.
        table_id (str): The ID of the table.
        experiment_name (str): The name of the experiment.

    Returns:
        list: List of unique sensor names.
    """
    query = f"""
    SELECT DISTINCT SensorData_Name
    FROM `iucc-f4d.{dataset_id}.{table_id}`
    WHERE ExperimentData_Exp_name = '{experiment_name}'
    ORDER BY SensorData_Name
    """
    
    query_job = client.query(query)
    results = query_job.result()
    
    return [row.SensorData_Name for row in results]

In [5]:
def query_table_name_and_email(client,dataset_id):
    query = f"""
    SELECT
      t2.table_name,
      STRING_AGG(t1.email, ', ') AS admin_emails
    FROM
      `iucc-f4d.user_device_permission.permissions` AS t1
    INNER JOIN
      `iucc-f4d.user_device_permission.mac_to_device` AS t2
      ON t1.mac_address = t2.mac_address
    WHERE
      t1.role = 'admin'
      AND t1.mac_address = '{dataset_id}'
    GROUP BY
      t2.table_name
    ORDER BY
      t2.table_name;
    """
  
    query_job = client.query(query)
    results = query_job.result()
    data = []
    return [row.admin_emails for row in results]

In [6]:
%%time
# query for the last 40 hours of data from all datasets and tables
big_df = pd.DataFrame()

for dataset in client.list_datasets():
    dataset_id = dataset.dataset_id
    for table in client.list_tables(dataset_id):
        if dataset_id == "user_device_permission":
            continue
        table_id = table.table_id
        try:
            print(f"Querying dataset: {dataset_id}, table: {table_id}")
            df = Last_Hours_Experiment_Data(client, dataset_id, table_id)
            big_df = pd.concat([big_df, df], ignore_index=True)
            print(f"Added data for {dataset_id}.{table_id}, rows: {len(df)}")
        except Exception as e:
            print(f"Error querying {dataset_id}.{table_id}: {e}")

# Add a new column for unique sensor name for entire experiment in big_df
for index, row in big_df.iterrows():
    dataset_id = row['dataset_id']
    table_id = row['table_id']
    experiment_name = row['ExperimentData_Exp_name']
    try:
        print(f"Processing row {index}: {dataset_id}, {table_id}, {experiment_name}")
        unique_sensors_entire_experiment = query_unique_sensors_name(client, dataset_id, table_id, experiment_name)
        ADNINS_EMAILS = query_table_name_and_email(client, table_id)
        if unique_sensors_entire_experiment:
            big_df.at[index, 'Unique_SensorData_Name'] = ', '.join(unique_sensors_entire_experiment)
            big_df.at[index, 'count_unique_sensor'] = len(unique_sensors_entire_experiment)
            if ADNINS_EMAILS:
                big_df.at[index, 'admin_emails'] = ADNINS_EMAILS[0]
            else:
                print(f"No admin emails found for dataset {dataset_id} and table {table_id}, adding default emails.")
                big_df.at[index, 'admin_emails'] = 'nir.averbuch@mail.huji.ac.il, idan.ifrach@mail.huji.ac.il, menachem.moshelion@mail.huji.ac.il, bnaya.hami@mail.huji.ac.il, Field4D_ADMIN@field4d.com'
    except Exception as e:
        print(f"Error processing row {index}: {e}")

Querying dataset: Icore_Pi, table: 2ccf6730ab5f
Added data for Icore_Pi.2ccf6730ab5f, rows: 2
Querying dataset: developerroom, table: 2ccf6730ab8c
Added data for developerroom.2ccf6730ab8c, rows: 0
Querying dataset: developerroom, table: d83adde26159
Added data for developerroom.d83adde26159, rows: 1
Querying dataset: idanhouse, table: 2ccf6774b0ae
Added data for idanhouse.2ccf6774b0ae, rows: 0
Querying dataset: menachem_moshelion, table: 2ccf6730ab7a
Added data for menachem_moshelion.2ccf6730ab7a, rows: 1
Querying dataset: menachem_moshelion, table: d83adde2608f
Added data for menachem_moshelion.d83adde2608f, rows: 1
Querying dataset: menachem_moshelion, table: d83adde261b0
Added data for menachem_moshelion.d83adde261b0, rows: 0
Querying dataset: yakir, table: d83adde260d1
Added data for yakir.d83adde260d1, rows: 0
Processing row 0: Icore_Pi, 2ccf6730ab5f, exp_7_KEREM_1
No admin emails found for dataset Icore_Pi and table 2ccf6730ab5f, adding default emails.
Processing row 1: Icore_Pi

## try send mail

In [ ]:
# GMAIL_USER="mosheliongreenhouse@gmail.com"
GMAIL_USER="f4d_support@field4d.com"
GMAIL_PASSWORD="eojevzaqemkqstic"
GMAIL_PASSWORD="wcog eelv clcp iaeo" # app password for mail

def send_email(to_addrs, subject, body, is_html=True):
    """
    Send an email using Gmail SMTP with credentials from environment variables.

    Args:
        to_addrs (str | list[str]): Recipient or list of recipients
        subject (str): Email subject
        body (str): Email content
        is_html (bool): True to send as HTML, False to send as plain text
    """
    if isinstance(to_addrs, str):
        to_addrs = [to_addrs]

    msg = MIMEMultipart("alternative")
    msg["From"] = GMAIL_USER
    msg["To"] = ", ".join(to_addrs)
    msg["Subject"] = subject

    mime_type = "html" if is_html else "plain"
    msg.attach(MIMEText(body, mime_type))

    with smtplib.SMTP("smtp.gmail.com", 587) as smtp:
        smtp.starttls()
        smtp.login(GMAIL_USER, GMAIL_PASSWORD)
        smtp.sendmail(GMAIL_USER, to_addrs, msg.as_string())

In [17]:
end_html_body = """
</table>
<p style="font-size: 14px; font-weight: bold; color: #000;">
<strong>Notice:</strong><br>
You are receiving this email because either there is an issue with the experiment or it has been concluded by the user.
</p>
If you notice missing sensors or delayed uploads, please review the sensor status at
<a href="https://field4d.com">https://field4d.com</a> and contact us if you have any questions.

<p style="font-size: 12px; color: #555; margin-top: 10px;">
<strong>About this report:</strong><br>
This table summarizes sensors that have not transmitted data recently.<br>
<ul style="margin-top: 5px; margin-bottom: 5px;">
  <li><strong>Experiment</strong> – The name of the experiment.</li>
  <li><strong>Dataset</strong> – The system name.</li>
  <li><strong>Last Timestamp</strong> – The most recent data received for this experiment.</li>
  <li><strong>Missing Sensors</strong> – Number of sensors in this experiment that have not sent data.</li>
  <li><strong>Hours Since Last Upload</strong> – Time elapsed since the last data sample was received <strong> from any sensor <strong>.</li>
</ul>
</p>
</body>
</html>
"""


In [19]:
def build_alerts_html(df):
    html = """
    <html>
    <head>
    <style>
    body { font-family: Arial, sans-serif; font-size: 13px; }
    table { border-collapse: collapse; width: 100%; font-size: 13px; }
    th, td { border: 1px solid #ddd; padding: 8px; }
    th { background-color: #4CAF50; color: white; }
    .missing { color: red; font-weight: bold; }
    .delayed { color: orange; font-weight: bold; }
    </style>
    </head>
    <body>
    <table>
    <tr>
        <th>Experiment</th>
        <th>Dataset</th>
        <th>Table</th>
        <th>Last Timestamp</th>
        <th>Missing Sensors</th>
        <th>Hours Since Last Upload</th>
    </tr>
    """
    
    # Sort the DataFrame by 'hours_passed_since_last_upload' (descending) and 'count_unique_sensor' (descending)
    df = df.sort_values(by=['hours_passed_since_last_upload', 'count_unique_sensor'], ascending=[False, False])
    

    for _, row in df.iterrows():
        missing_sensors = int(row["count_unique_sensor"] - row["Sensor_In_Last_40h"])
        delayed_class = "delayed" if row["hours_passed_since_last_upload"] > 2 else ""
        missing_class = "missing" if missing_sensors > 0 else ""

        html += f"""
        <tr>
            <td>{row['ExperimentData_Exp_name']}</td>
            <td>{row['dataset_id']}</td>
            <td>{row['table_id']}</td>
            <td>{row['last_timestamp']}</td>
            <td class="{missing_class}">{missing_sensors} / {int(row['count_unique_sensor'])}</td>
            <td class="{delayed_class}">{row['hours_passed_since_last_upload']}</td>
        </tr>
        """
    return html


def send_alerts_by_admin(df):
    # Filter only problematic rows
    problem_df = df[
        (df["hours_passed_since_last_upload"] > 2) |
        (df["count_unique_sensor"] > df["Sensor_In_Last_40h"])
    ]

    # Group by dataset_id
    for dataset_group, group_df in problem_df.groupby("dataset_id"):
        # Get unique admin_emails for this dataset group
        admin_emails = group_df["admin_emails"].unique()
        # Flatten and deduplicate emails
        all_emails = set()
        for emails in admin_emails:
            all_emails.update([email.strip() for email in emails.split(",")])
        to_addrs = list(all_emails)
        to_addrs = ["nir.averbuch@mail.huji.ac.il"]
        html_body = build_alerts_html(group_df)
        html_body += end_html_body
        send_email(
            to_addrs=to_addrs,
            subject=f"⚠ Sensor Upload Alert for {dataset_group}",
            body=html_body,
            is_html=True
        )

In [20]:
send_alerts_by_admin(big_df)